# Fine-tune a Philosopher Bot (Free, Google Colab)

This notebook fine-tunes an open-weight model on your `dialogue_pairs.jsonl` dataset using **QLoRA** via **Unsloth** — free on Colab's T4 GPU.

**Before running:**
1. In Colab: `Runtime` -> `Change runtime type` -> select **T4 GPU** (free tier)
2. Upload your `dialogue_pairs.jsonl` file (from `extract_dialogue_dataset.py`) using the file browser on the left, or mount Google Drive
3. Run cells top to bottom

**Time estimate:** ~30-90 minutes depending on dataset size and Colab's current GPU availability.

In [ ]:
# Install Unsloth and dependencies (free, open source)
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir --no-deps unsloth unsloth_zoo

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None          # auto-detect (float16 for T4)
load_in_4bit = True   # QLoRA -- this is what makes it fit on a free GPU

# You can swap this for another open-weight base model, e.g.
# "unsloth/mistral-7b-v0.3-bnb-4bit" or "unsloth/Qwen2.5-7B-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3.1-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# Attach LoRA adapters -- these small trainable layers are what actually
# get updated. The base model's weights stay frozen, which is why this
# is cheap and fast.
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                     # LoRA rank -- higher = more capacity, slower
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# Upload dialogue_pairs.jsonl here via the Colab file browser (left sidebar),
# or mount Drive and point to it there. Update the path below if needed.
DATASET_PATH = "dialogue_pairs.jsonl"

from datasets import load_dataset

raw_dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Loaded {len(raw_dataset)} training pairs")
print(raw_dataset[0])

In [ ]:
# Format each pair into a chat-style prompt the model trains on.
# This teaches the model the Socratic argumentative rhythm: a claim or
# question, followed by a philosophical response.

PROMPT_TEMPLATE = """Below is a philosophical exchange. Continue it as {speaker_response} would, reasoning carefully and in character.

{speaker_prompt}: {instruction}

{speaker_response}:"""

EOS_TOKEN = tokenizer.eos_token

def format_example(example):
    prompt = PROMPT_TEMPLATE.format(
        speaker_response=example["speaker_response"],
        speaker_prompt=example["speaker_prompt"],
        instruction=example["instruction"],
    )
    text = prompt + " " + example["output"] + EOS_TOKEN
    return {"text": text}

dataset = raw_dataset.map(format_example)
print(dataset[0]["text"][:500])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,          # increase if you have a small dataset
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

In [ ]:
# This is the actual training run. Watch the loss column -- it should
# generally trend down. Colab will disconnect if idle too long, so keep
# the tab active or check back periodically.
trainer_stats = trainer.train()

In [ ]:
# Quick sanity check: ask it something and see how it responds
FastLanguageModel.for_inference(model)

test_prompt = PROMPT_TEMPLATE.format(
    speaker_response="Socrates",
    speaker_prompt="Glaucon",
    instruction="Is justice simply the advantage of the stronger?",
)

inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200, use_cache=True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

## Export for local use with Ollama

This converts your fine-tuned model to GGUF format so you can run it
locally with Ollama, exactly like your base model — free forever, no
cloud costs after this point.

In [ ]:
# Save merged model as GGUF (quantized) -- this downloads to your Colab
# session; download it from the file browser afterward, or save directly
# to Google Drive by mounting it first.
model.save_pretrained_gguf("philosopher-model", tokenizer, quantization_method="q4_k_m")

print("Done. Download the .gguf file from the Colab file browser (left sidebar).")
print("Then on your local machine:")
print("  1. Create a file named 'Modelfile' with: FROM ./your-model.gguf")
print("  2. Run: ollama create philosopher-custom -f Modelfile")
print("  3. Update OLLAMA_MODEL in query.py to 'philosopher-custom'")